# Q5 - Análise de Clientes (Pipeline Completo)

In [8]:
import pandas as pd

In [9]:
# Carregar dados
vendas = pd.read_csv('../data/raw/vendas_2023_2024.csv')
produtos = pd.read_csv('../data/processed/produtos_clean.csv')

print(vendas.head())
print(produtos.head())

   id  id_client  id_product  qtd     total   sale_date
0   0         42         105   11    3405.0  2023-09-10
1   1          3         136    9   16873.9  15-09-2024
2   2         25         139    7    9475.3  2024-08-13
3   4         20          23    5   55893.0  2023-02-03
4   5          8          57    4  451403.9  2024-02-12
                             name    price  code actual_category
0     Transponder AIS Maré Magnum  3312252     1     eletrônicos
1       Transponder Furuno Marlin  1399815     2     eletrônicos
2    Radar Furuno Pulse Leviathan   902419     3     eletrônicos
3       Rádio AIS Hydro Tidal Zen   338188     4       ancoragem
4  Piloto Automático Furuno Storm  2366901     5     eletrônicos


In [21]:
# Padronização nomes de colunas
if 'id_product' in vendas.columns:
    vendas = vendas.rename(columns={'id_product': 'product_id'})

if 'code' in produtos.columns:
    produtos = produtos.rename(columns={'code': 'product_id'})

In [24]:
assert 'product_id' in vendas.columns
assert 'product_id' in produtos.columns

In [25]:
# Merge vendas + produtos
df = vendas.merge(produtos, on='product_id', how='left')

print(df.head())

   id  id_client  product_id  qtd     total   sale_date  \
0   0         42         105   11    3405.0  2023-09-10   
1   1          3         136    9   16873.9  15-09-2024   
2   2         25         139    7    9475.3  2024-08-13   
3   4         20          23    5   55893.0  2023-02-03   
4   5          8          57    4  451403.9  2024-02-12   

                                       name     price actual_category  
0              Cabo de Nylon Danforth Prime     30954       ancoragem  
1            Cabo de Nylon Bruce Flux Hydro     19735       ancoragem  
2       Boia de Arqueamento Danforth Torque    142488       ancoragem  
3      Piloto Automático Furuno Torque Peak   1117863     eletrônicos  
4  Motor de Popa Honda Vector Kinetic 174HP  11879057       propulsão  


In [26]:
print(vendas.columns)
print(produtos.columns)

Index(['id', 'id_client', 'product_id', 'qtd', 'total', 'sale_date'], dtype='object')
Index(['name', 'price', 'product_id', 'actual_category'], dtype='object')


In [27]:
# Métricas por cliente
clientes = (
    df.groupby('id_client')
    .agg(
        faturamento_total=('total', 'sum'),
        frequencia=('id_client', 'count'),
        diversidade_categorias=('actual_category', 'nunique')
    )
)

clientes['ticket_medio'] = clientes['faturamento_total'] / clientes['frequencia']

clientes.head()

,faturamento_total,frequencia,diversidade_categorias,ticket_medio
id_client,,,,
1,51092500.05,190,3,268907.895000
2,65652931.35,220,3,298422.415227
3,59575349.10,207,3,287803.618841
4,50691754.40,207,3,244887.702415
5,58592802.70,202,3,290063.379703


In [32]:
import pandas as pd

# =========================
# 1. Carregar dados
# =========================
vendas = pd.read_csv('../data/raw/vendas_2023_2024.csv')
produtos = pd.read_csv('../data/processed/produtos_clean.csv')

# =========================
# 2. Padronização de colunas
# =========================
if 'id_product' in vendas.columns:
    vendas = vendas.rename(columns={'id_product': 'product_id'})

if 'code' in produtos.columns:
    produtos = produtos.rename(columns={'code': 'product_id'})

# =========================
# 3. Merge vendas + produtos
# =========================
df = vendas.merge(produtos, on='product_id', how='left')

# =========================
# 4. Métricas por cliente
# =========================
clientes = (
    df.groupby('id_client')
    .agg(
        faturamento_total=('total', 'sum'),
        frequencia=('id_client', 'count'),
        diversidade_categorias=('actual_category', 'nunique')
    )
)

# Ticket médio
clientes['ticket_medio'] = (
    clientes['faturamento_total'] / clientes['frequencia']
)

# =========================
# 5. Filtro clientes elite (>=3 categorias)
# =========================
clientes_elite = clientes[clientes['diversidade_categorias'] >= 3]

# Ordenação (desempate por id_client)
clientes_elite = clientes_elite.sort_values(
    by=['ticket_medio', 'id_client'],
    ascending=[False, True]
)

# Top 10 clientes
top10 = clientes_elite.head(10)

print("\nTop 10 clientes (dados brutos):")
print(top10)

# =========================
# 6. Filtrar compras dos Top 10
# =========================
df_top10 = df[df['id_client'].isin(top10.index)]

# =========================
# 7. Categoria mais vendida
# =========================
categoria_top = (
    df_top10.groupby('actual_category')['qtd']
    .sum()
    .sort_values(ascending=False)
)

categoria_mais_vendida = categoria_top.idxmax()
total_itens = categoria_top.max()

print("\nCategoria mais vendida:")
print(f"{categoria_mais_vendida} ({total_itens} itens)")

# =========================
# 8. Formatação (APENAS VISUAL)
# =========================
top10_formatado = top10.copy()

def formatar_brl(valor):
    return f"R$ {valor:,.2f}"

top10_formatado['faturamento_total'] = top10_formatado['faturamento_total'].apply(formatar_brl)
top10_formatado['ticket_medio'] = top10_formatado['ticket_medio'].apply(formatar_brl)

print("\nTop 10 clientes (formatado):")
print(top10_formatado)


Top 10 clientes (dados brutos):
           faturamento_total  frequencia  diversidade_categorias  \
id_client                                                          
47               64003343.75         190                       3   
42               72187369.50         222                       3   
9                66788855.35         218                       3   
22               59581398.75         198                       3   
2                65652931.35         220                       3   
28               60826837.25         204                       3   
46               59126834.35         199                       3   
38               57093331.15         195                       3   
36               62791038.15         215                       3   
5                58592802.70         202                       3   

            ticket_medio  
id_client                 
47         336859.703947  
42         325168.331081  
9          306370.896101  
22         3009

In [29]:
# Filtrar compras dos Top 10
df_top10 = df[df['id_client'].isin(top10.index)]

df_top10.head()

,id,id_client,product_id,qtd,total,sale_date,name,price,actual_category
0,0,42,105,11,3405.0,2023-09-10,Cabo de Nylon Danforth Prime,30954,ancoragem
5,6,36,52,3,39056.4,2023-09-26,Motor Diesel Honda Zen Tidal Mako 26HP,137041,propulsão
28,30,38,35,15,165083.0,2023-07-18,Radar Furuno Swift,1100553,eletrônicos
32,34,28,73,3,429480.0,05-11-2023,Motor de Popa Torqeedo Core Hydra Flux 162HP,14315993,propulsão
36,38,38,87,9,601739.0,2024-04-30,Motor Elétrico Volvo Boost 225HP,668599,propulsão


In [30]:
# Categoria mais vendida (quantidade)
categoria_top = (
    df_top10.groupby('actual_category')['qtd']
    .sum()
    .sort_values(ascending=False)
)

categoria_top

actual_category
ancoragem      6198
propulsão      6030
eletrônicos    4648
Name: qtd, dtype: int64

In [31]:
# Resultado final
categoria_mais_vendida = categoria_top.idxmax()
total_itens = categoria_top.max()

print(f"Categoria mais vendida: {categoria_mais_vendida}")
print(f"Total de itens vendidos: {total_itens}")

Categoria mais vendida: ancoragem
Total de itens vendidos: 6198
